In [ ]:
#bliblioteca para manipulação de dados e leitura de arquivos CSV
import pandas as pd
import numpy as np
df = pd.read_csv('../data/bronze.csv')

In [ ]:
# Loop para imprimir os valores únicos de cada coluna e possibilitar analises
for coluna in df.columns:
    print(f"Valores únicos na coluna '{coluna}':")
    print(df[coluna].unique())
    print("-" * 30)

Valores únicos na coluna 'rating':
['A-' 'AAA' 'BBB-' 'AA-' 'BBB' 'BB' 'A' 'B' 'BBB+' 'BB+' 'B+' 'BB-' 'B-'
 'A+' 'CCC' 'AA' 'CCC+' 'CC' 'CCC-' 'AA+' 'C' 'D']
------------------------------
Valores únicos na coluna 'current_ratio':
[1.1507 1.1129 1.9276 ... 1.4905 2.1505 3.4927]
------------------------------
Valores únicos na coluna 'long_term_debt_per_capital':
[0.4551 0.0072 0.2924 ... 0.7713 0.8036 0.5065]
------------------------------
Valores únicos na coluna 'debtperequity_ratio':
[0.8847 0.0073 0.4255 ... 3.3729 4.462  1.0263]
------------------------------
Valores únicos na coluna 'gross_margin':
[77.623  43.6619 11.9008 ... 45.0411 63.5257 79.3878]
------------------------------
Valores únicos na coluna 'operating_margin':
[19.4839 19.8327  3.3173 ...  1.0137 11.4376 25.5703]
------------------------------
Valores únicos na coluna 'ebit_margin':
[19.4839 19.8327  3.3173 ...  1.0137 14.0399 25.5703]
------------------------------
Valores únicos na coluna 'ebitda_margin':
[28.9

In [ ]:
#dropar duplicatas para evitar que o modelo aprenda padrões errados ou seja influenciado por dados redundantes.
df = df.drop_duplicates(subset=["name", "domain", "employees", "long_term_debt_per_capital", "debtperequity_ratio",
                                "gross_margin","operating_margin", "ebit_margin"], keep="first")

In [ ]:
#dropar colunas de apenas texto e labels que não são uteis ou são repetidas.
colunas_para_dropar = ['rating_agency', 'corporation', 'rating_date', 'cik', 'binary_rating', 'sic_code', 'sector_x',
                       'involvement', 'involvement_msci', 'domain', 'ticker']
df = df.drop(columns=colunas_para_dropar)

In [ ]:
# verificação de valores ausentes para tratamento posterior
print(df.isna().sum())

for col in df.columns:
    if df[col].isna().sum() > 0:
        print(f"Coluna '{col}' tem {df[col].isna().sum()} valores ausentes.")

rating                                      0
current_ratio                               0
long_term_debt_per_capital                  0
debtperequity_ratio                         0
gross_margin                                0
                                         ... 
esg                                       841
involvement_msci_controversial_weapons      0
involvement_msci_gambling                   0
involvement_msci_tobacco_products           0
involvement_msci_alcoholic_beverages        0
Length: 69, dtype: int64
Coluna 'employees' tem 175 valores ausentes.
Coluna 'altman_score' tem 1065 valores ausentes.
Coluna 'piotroski_score' tem 951 valores ausentes.
Coluna 'controversies_environment' tem 936 valores ausentes.
Coluna 'controversies_social' tem 936 valores ausentes.
Coluna 'controversies_customers' tem 936 valores ausentes.
Coluna 'controversies_human_rights_and_community' tem 936 valores ausentes.
Coluna 'controversies_labor_rights_and_supply_chain' tem 936 valores aus

In [ ]:
# verificar tipo de dados de cada coluna para tratamento posterior
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2834 entries, 0 to 8663
Data columns (total 69 columns):
 #   Column                                                             Non-Null Count  Dtype  
---  ------                                                             --------------  -----  
 0   rating                                                             2834 non-null   object 
 1   current_ratio                                                      2834 non-null   float64
 2   long_term_debt_per_capital                                         2834 non-null   float64
 3   debtperequity_ratio                                                2834 non-null   float64
 4   gross_margin                                                       2834 non-null   float64
 5   operating_margin                                                   2834 non-null   float64
 6   ebit_margin                                                        2834 non-null   float64
 7   ebitda_margin                

Storm da logica de tratamento

[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,32,33,34,41,46,64] ja são numeros
[42,43] porcentagem limpar falhas e colocar _pct (porcentagem) para lembrança, limpar pra float
[0] notas mapear logica de conversão para numeros
[17] coluna para identificação (name)
[18,19,20,21,22,23,24,25,26,27,28,29,30,31,44,45,65,66,67,68]mapear logica de conversão para numeros (2,1,0) yes-no-unknow
[35,36,37,38,39,40] mapear logica de conversão para numeros: cores (4,3,2,1,0) Green-Yellow-Orange-Red-unknow
[47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63] mapear logica de conversão para numeros: (3,2,1,0) strongalined-aligned-no-unknow

todos os campos que estiverem vazio viram unknow
todos os unknow vão ser 0

In [ ]:
#mudar tipo de dados por coluna [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,32,33,34,41,46,64] para float
colflo = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,32,33,34,41,46,64]

for col in colflo:
    df[df.columns[col]] = df[df.columns[col]].astype(float)

In [ ]:
#ver nome das colunas
print(df.columns[42:44])

Index(['decarbonization_target_comprehensiveness', 'decarbonization_target_ambition_p_a'], dtype='object')


In [ ]:
#renomear colunas que tinham numeros com % pra lembrar que são %porcentagem apos tratamento
df.rename(columns={"decarbonization_target_comprehensiveness": "decarbonization_target_comprehensiveness_pct"}, inplace=True)
df.rename(columns={"decarbonization_target_ambition_p_a": "decarbonization_target_ambition_p_a_pct"}, inplace=True)

In [ ]:
#ver dados da coluna e entender a necessidade de tratamento
print(df["decarbonization_target_ambition_p_a_pct"].unique())

[nan '-3.57%' '0.39%' '-0.53%' '0.79%' '-5.56%' '-3.79%' '0.94%' '-1.73%'
 '-2.64%' '-0.02%' '0.57%' '0.65%' '-4.35%' '1.04%' '0.29%' '-2.97%'
 '0.91%' '0.74%' '-2.35%' '0.8%' '-1.06%' '1.0%' '0.82%' '-7.14%' '0.2%'
 '-3.45%' '0.87%' '0.92%' '-0.26%' '-0.13%' '-0.15%' '0.06%' '-8.12%'
 '-7.92%' '-1.33%' '-10.2%' '-2.34%' '-0.97%' '0.83%' '0.72%' '-3.14%'
 '-1.13%' '-1.75%' '0.66%' '-1.16%' '-2.71%' '-1.28%' '0.49%' '1.02%'
 '-8.48%' '-0.68%' '0.95%' '-5.37%' '-3.05%' '-0.25%' '-0.86%' '0.54%'
 '-2.77%' '-0.52%' '-2.7%' '0.28%' '-2.02%' '-5.16%' '0.86%' '-5.26%'
 '0.47%' '-2.75%' '-2.85%' '-0.36%' '0.5%' '-0.32%' '-0.44%' '0.97%'
 '0.22%' '0.04%' '0.1%' '-1.91%' '-1.1%' '-6.79%' '-0.64%' '1.05%'
 '-5.32%' '-0.31%' '0.03%' '-7.69%' '-0.19%' '0.93%' '-4.87%' '-3.33%'
 '-1.63%' '0.75%' '0.85%' '-1.59%' '0.08%' '-2.21%' '-4.71%' '-2.61%'
 '-3.83%' '-12.5%' '-2.76%' '-3.23%' '-0.9%' '-0.45%' '-3.39%' '-2.42%'
 '-4.21%' '-0.46%' '0.88%' '-5.44%' '-1.44%' '-3.27%' '-0.87%' '-0.07%'
 '0.89%' '0

In [ ]:
#remover porcentagem mantendo numeros  negativo
df["decarbonization_target_ambition_p_a_pct"] = (
    df["decarbonization_target_ambition_p_a_pct"]
    .str.replace(r"[^0-9.-]", "", regex=True)
    .astype(float)
)

In [ ]:
#ver os dados unicos da coluna para entender a necessidade de tratamento
print(df["decarbonization_target_comprehensiveness_pct"].unique())

[nan '100.0%' '16.63%' '\\n6.82%' '\\n7.52%' '\\n1.93%' '73.78%' '80.18%'
 '\\n4.95%' '\\n6.38%' '\\n5.59%' '\\n2.17%' '34.88%' '63.33%' '\\n5.07%'
 '\\n3.86%' '99.96%' '\\n3.51%' '50.22%' '\\n3.19%' '\\n8.72%' '\\n6.93%'
 '11.11%' '\\n5.08%' '69.62%' '\\n3.09%' '\\n4.91%' '\\n0.62%' '28.74%'
 '\\n8.79%' '23.15%' '\\n95.6%' '68.56%' '\\n54.0%' 't\\n2.8%' '\\n5.61%'
 '45.47%' '58.36%' '\\n5.93%' '48.97%' '81.73%' '53.97%' '\\n9.05%'
 '1.58%' '\\n1.58%' '48.16%' '\\n1.31%' '66.51%' '29.62%' '\\n30.4%'
 '\\n3.66%' '96.12%' '81.57%' '\\n50.3%' '97.89%' '\\n0.52%' '12.45%'
 '\\n0.77%' '\\n82.5%' '84.62%' '36.73%' '\\n9.59%' '33.63%' '\\n2.04%'
 '13.43%' '\\n3.82%' '53.14%' '14.82%' '\\n9.21%' '46.87%' '42.05%'
 '\\n3.75%' '91.55%' '11.29%' '\\n96.5%' '\\n3.87%' '29.15%' '\\n4.65%'
 '\\n0.64%' '43.87%' '\\n96.8%' '36.58%' '\\n8.13%' '\\n6.24%' '\\n9.84%'
 '97.94%' '10.72%' '\\n0.35%' '99.91%' '\\n2.89%' '\\n0.58%' '81.44%'
 '90.37%' '82.89%' '92.83%' '50.13%' '29.77%' '77.05%' '98.74%' '37.4

In [ ]:
#remover porcentagem e outras falhas mantendo apenas numeros e ponto decimal
df["decarbonization_target_comprehensiveness_pct"] = (
    df["decarbonization_target_comprehensiveness_pct"]
    .str.replace(r"[^0-9.]", "", regex=True)
    .astype(float)
)

In [ ]:
# Preencher valores ausentes com 'unknown'
df.fillna("unknown", inplace=True)

C:\Users\mikae\AppData\Local\Temp\ipykernel_30852\2174511668.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'unknown' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("unknown", inplace=True)


In [ ]:
# preencher colunas com repostas 'yes', 'no' e 'unknown' para 2, 1 e 0 respectivamente
cols = [18,19,20,21,22,23,24,25,26,27,28,29,30,31,44,45,65,66,67,68]

map_values = {
    "yes": 2,
    "no": 1,
    "unknown": 0
}

df[df.columns[cols]] = df[df.columns[cols]].astype(object)

df[df.columns[cols]] = df[df.columns[cols]].apply(
    lambda c: c.astype(str).str.strip().str.lower().map(map_values)
)

df[df.columns[cols]] = df[df.columns[cols]].fillna(0).astype("int64")

In [ ]:
# alterar os valores das colunas com dados em cores 'green', 'yellow', 'orange', 'red' e 'unknown' para 4, 3, 2, 1 e 0 respectivamente
cols = [35,36,37,38,39,40]

map_values = {
    "green": 4,
    "yellow": 3,
    "orange": 2,
    "red": 1,
    "unknown": 0
}

df[df.columns[cols]] = df[df.columns[cols]].astype(object)

df[df.columns[cols]] = df[df.columns[cols]].apply(
    lambda c: c.astype(str).str.strip().str.lower().map(map_values)
)

df[df.columns[cols]] = df[df.columns[cols]].fillna(0).astype("int64")

In [ ]:
# alterar os valores das colunas de alinhamento para ordem numerica.
cols = [47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63]

map_values = {
    "strongly_aligned": 3,
    "aligned": 2,
    "no": 1,
    "unknown": 0
}

df[df.columns[cols]] = df[df.columns[cols]].astype(object)

df[df.columns[cols]] = df[df.columns[cols]].apply(
    lambda c: c.astype(str).str.strip().str.lower().map(map_values)
)

df[df.columns[cols]] = df[df.columns[cols]].fillna(0).astype("int64")

In [ ]:
# ver dados da coluna de notas para entender a necessidade de tratamento
print(df["rating"].unique())

['A-' 'AAA' 'BBB-' 'AA-' 'BBB' 'BB' 'A' 'B' 'BBB+' 'BB+' 'B+' 'BB-' 'B-'
 'A+' 'CCC' 'AA' 'CCC+' 'CC' 'CCC-' 'AA+' 'C' 'D']


In [ ]:
#mapeamento e conversao de notas para ordem numerica
import pandas as pd

ratings_order = [
    "D","C","CC","CCC-","CCC","CCC+",
    "B-","B","B+","BB-","BB","BB+",
    "BBB-","BBB","BBB+",
    "A-","A","A+",
    "AA-","AA","AA+",
    "AAA"
]

rating_map = {r: i for i, r in enumerate(ratings_order)}

df["rating"] = (
    df["rating"]
    .astype(str)
    .str.strip()
    .str.upper()
    .map(rating_map)
)

In [ ]:
#verificar o resultado do mapeamento e conversão
print(rating_map)
{'D': 0, 'C': 1, 'CC': 2, 'CCC-': 3, 'CCC': 4, 'CCC+': 5, 'B-': 6, 'B': 7, 'B+': 8, 'BB-': 9, 'BB': 10, 'BB+': 11,
  'BBB-': 12, 'BBB': 13, 'BBB+': 14, 'A-': 15, 'A': 16, 'A+': 17, 'AA-': 18, 'AA': 19, 'AA+': 20, 'AAA': 21}

{'D': 0, 'C': 1, 'CC': 2, 'CCC-': 3, 'CCC': 4, 'CCC+': 5, 'B-': 6, 'B': 7, 'B+': 8, 'BB-': 9, 'BB': 10, 'BB+': 11, 'BBB-': 12, 'BBB': 13, 'BBB+': 14, 'A-': 15, 'A': 16, 'A+': 17, 'AA-': 18, 'AA': 19, 'AA+': 20, 'AAA': 21}


{'D': 0,
 'C': 1,
 'CC': 2,
 'CCC-': 3,
 'CCC': 4,
 'CCC+': 5,
 'B-': 6,
 'B': 7,
 'B+': 8,
 'BB-': 9,
 'BB': 10,
 'BB+': 11,
 'BBB-': 12,
 'BBB': 13,
 'BBB+': 14,
 'A-': 15,
 'A': 16,
 'A+': 17,
 'AA-': 18,
 'AA': 19,
 'AA+': 20,
 'AAA': 21}

In [ ]:
# trocar ultimos unknown que sobraram para 0
df = df.replace("unknown", 0)

C:\Users\mikae\AppData\Local\Temp\ipykernel_30852\1732595094.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("unknown", 0)


In [ ]:
#baixar o arquivo tratado final para uso no treino do modelo
df.to_csv(r"C:\\Users\\mikae\\OneDrive\\Área de Trabalho\\CESAR\\Projetos 3\\dados_tratadosfim.csv", index=False)

NA HIPOTESE DE TREINAR O MODELO COM MAIS DADOS EMBORA ALGUNS DUPLICADOS DESCOMENTE OS ELEMENTOS ABAIXO

In [ ]:
# #bliblioteca para manipulação de dados e leitura de arquivos CSV
# import pandas as pd
# import numpy as np
# df = pd.read_csv("C:\\Users\\mikae\\OneDrive\\Área de Trabalho\\CESAR\\Projetos 3\\bronze.csv")

In [ ]:
# #dropar colunas de apenas texto e labels que não são uteis ou são repetidas.
# colunas_para_dropar = ['rating_agency', 'corporation', 'rating_date', 'cik', 'binary_rating', 'sic_code', 'sector_x',
#                        'involvement', 'involvement_msci', 'domain', 'ticker']
# df = df.drop(columns=colunas_para_dropar)

In [ ]:
# # verificação de valores ausentes para tratamento posterior
# print(df.isna().sum())

# for col in df.columns:
#     if df[col].isna().sum() > 0:
#         print(f"Coluna '{col}' tem {df[col].isna().sum()} valores ausentes.")

In [ ]:
# # verificar tipo de dados de cada coluna para tratamento posterior
# df.info()

In [ ]:
# # mudar tipo de dados por coluna [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,32,33,34,41,46,64] para float
# colflo = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,32,33,34,41,46,64]

# for col in colflo:
#     df[df.columns[col]] = df[df.columns[col]].astype(float)

In [ ]:
# #renomear colunas que tinham numeros com % pra lembrar que são %porcentagem apos tratamento
# df.rename(columns={"decarbonization_target_comprehensiveness": "decarbonization_target_comprehensiveness_pct"}, inplace=True)
# df.rename(columns={"decarbonization_target_ambition_p_a": "decarbonization_target_ambition_p_a_pct"}, inplace=True)

In [ ]:
# #ver dados da coluna e entender a necessidade de tratamento
# print(df["decarbonization_target_ambition_p_a_pct"].unique())

In [ ]:
# #remover porcentagem mantendo numeros negativo
# df["decarbonization_target_ambition_p_a_pct"] = (
#     df["decarbonization_target_ambition_p_a_pct"]
#     .str.replace(r"[^0-9.-]", "", regex=True)
#     .astype(float)
# )

In [ ]:
# #ver os dados unicos da coluna para entender a necessidade de tratamento
# print(df["decarbonization_target_comprehensiveness_pct"].unique())

In [ ]:
# #remover porcentagem e outras falhas mantendo apenas numeros e ponto decimal
# df["decarbonization_target_comprehensiveness_pct"] = (
#     df["decarbonization_target_comprehensiveness_pct"]
#     .str.replace(r"[^0-9.]", "", regex=True)
#     .astype(float)
# )

In [ ]:
# # Preencher valores ausentes das colunas com 'unknown'
# df = df.fillna('unknown')

In [ ]:
# # alterar os valores das colunas de alinhamento para ordem numerica.
# cols = [47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63]

# map_values = {
#     "strongly_aligned": 3,
#     "aligned": 2,
#     "no": 1,
#     "unknown": 0
# }

# df[df.columns[cols]] = df[df.columns[cols]].astype(object)

# df[df.columns[cols]] = df[df.columns[cols]].apply(
#     lambda c: c.astype(str).str.strip().str.lower().map(map_values)
# )

# df[df.columns[cols]] = df[df.columns[cols]].fillna(0).astype("int64")

In [ ]:
# # alterar os valores das colunas com dados em cores 'green', 'yellow', 'orange', 'red' e 'unknown' para 4, 3, 2, 1 e 0 respectivamente
# cols = [35,36,37,38,39,40]

# map_values = {
#     "green": 4,
#     "yellow": 3,
#     "orange": 2,
#     "red": 1,
#     "unknown": 0
# }

# df[df.columns[cols]] = df[df.columns[cols]].astype(object)

# df[df.columns[cols]] = df[df.columns[cols]].apply(
#     lambda c: c.astype(str).str.strip().str.lower().map(map_values)
# )

# df[df.columns[cols]] = df[df.columns[cols]].fillna(0).astype("int64")

In [ ]:
# # preencher colunas com repostas 'yes', 'no' e 'unknown' para 2, 1 e 0 respectivamente
# cols = [18,19,20,21,22,23,24,25,26,27,28,29,30,31,44,45,65,66,67,68]

# map_values = {
#     "yes": 2,
#     "no": 1,
#     "unknown": 0
# }

# df[df.columns[cols]] = df[df.columns[cols]].astype(object)

# df[df.columns[cols]] = df[df.columns[cols]].apply(
#     lambda c: c.astype(str).str.strip().str.lower().map(map_values)
# )

# df[df.columns[cols]] = df[df.columns[cols]].fillna(0).astype("int64")

In [ ]:
# # ver dados da coluna de notas para entender a necessidade de tratamento
# print(df["rating"].unique())

In [ ]:
# #mapeamento e conversao de notas para ordem numerica
# import pandas as pd

# ratings_order = [
#     "D","C","CC","CCC-","CCC","CCC+",
#     "B-","B","B+","BB-","BB","BB+",
#     "BBB-","BBB","BBB+",
#     "A-","A","A+",
#     "AA-","AA","AA+",
#     "AAA"
# ]

# rating_map = {r: i for i, r in enumerate(ratings_order)}

# df["rating"] = (
#     df["rating"]
#     .astype(str)
#     .str.strip()
#     .str.upper()
#     .map(rating_map)
# )

In [ ]:
# #verificar o resultado do mapeamento e conversão
# print(rating_map)
# {'D': 0, 'C': 1, 'CC': 2, 'CCC-': 3, 'CCC': 4, 'CCC+': 5, 'B-': 6, 'B': 7, 'B+': 8, 'BB-': 9, 'BB': 10, 'BB+': 11,
#   'BBB-': 12, 'BBB': 13, 'BBB+': 14, 'A-': 15, 'A': 16, 'A+': 17, 'AA-': 18, 'AA': 19, 'AA+': 20, 'AAA': 21}

In [ ]:
# # trocar ultimos unknown que sobraram para 0
# df = df.replace(to_replace=r"(?i)^unknown$", value=0, regex=True)

In [ ]:
# # troca de tipos dos dados restantes de int pra float
# fimflo = list(range(-1)) + list(range(18,32)) + list(range(35,42)) + list(range(44,46)) + list(range(47,64)) + list(range(65,69))

# for col in fimflo:
#     df[df.columns[col]] = df[df.columns[col]].astype(float)

In [ ]:
# # trocar ultimos unknown que sobraram para 0
# df = df.replace(to_replace=r"(?i)^unknown$", value=0, regex=True)

In [ ]:
# #baixar o arquivo fillado final para uso no treino do modelo
# df.to_csv(r"C:\\Users\\mikae\\OneDrive\\Área de Trabalho\\CESAR\\Projetos 3\\dados_testeall.csv", index=False)